# DDRI Bike-Change rep15 군집 포함 효과 비교

## 용어 설명

- `cluster 포함`: 입력 변수에 `cluster(군집 라벨)`을 넣는 경우
- `cluster 제외`: 같은 피처 세트에서 `cluster`만 뺀 경우
- `sign_accuracy`(부호 일치율): 예측값과 실제값의 증가/감소 방향이 같은 비율

목표:
- `bike_change` 기준선에서 군집 라벨이 실제로 도움이 되는지 직접 비교한다.


In [1]:
from pathlib import Path

import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path('/Users/cheng80/Desktop/ddri_work')
WORK_DIR = ROOT / 'works/07_prediction_bike_change'
OUTPUT_DATA_DIR = WORK_DIR / 'output/data'

TRAIN_PATH = OUTPUT_DATA_DIR / 'ddri_prediction_bike_change_train_2023_2024.csv'
TEST_PATH = OUTPUT_DATA_DIR / 'ddri_prediction_bike_change_test_2025.csv'
METRICS_PATH = OUTPUT_DATA_DIR / 'ddri_bike_change_rep15_cluster_effect_metrics.csv'


In [2]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

FEATURE_COLUMNS_WITH_CLUSTER = [
    'station_id', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday',
    'temperature', 'humidity', 'precipitation', 'wind_speed',
    'lag_1h', 'lag_24h', 'lag_168h', 'rolling_mean_24h', 'rolling_std_24h',
    'rolling_mean_168h', 'rolling_std_168h', 'rolling_mean_6h',
    'is_weekend', 'is_night_hour', 'is_commute_hour', 'hour_sin', 'is_rainy', 'hour_cos',
    'commute_morning_flag', 'commute_evening_flag', 'subway_distance_m', 'distance_naturepark_m',
    'restaurant_count_300m', 'convenience_store_count_300m', 'bus_stop_count_300m', 'cafe_count_300m',
    'elevation_diff_nearest_subway_m', 'nearest_park_area_sqm'
]
FEATURE_COLUMNS_NO_CLUSTER = [c for c in FEATURE_COLUMNS_WITH_CLUSTER if c != 'cluster']

CATEGORICAL_WITH_CLUSTER = ['station_id', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday']
CATEGORICAL_NO_CLUSTER = ['station_id', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday']

train_2023 = train_df[train_df['date'] < '2024-01-01'].copy()
valid_2024 = train_df[train_df['date'] >= '2024-01-01'].copy()
full_train = train_df.copy()


In [3]:
def evaluate_predictions(model_name: str, split_name: str, y_true: pd.Series, y_pred) -> dict:
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    sign_accuracy = ((pd.Series(y_pred) >= 0) == (y_true.reset_index(drop=True) >= 0)).mean()
    return {
        'model': model_name,
        'split': split_name,
        'rmse': round(float(rmse), 4),
        'mae': round(float(mae), 4),
        'r2': round(float(r2), 4),
        'sign_accuracy': round(float(sign_accuracy), 4),
    }

def prepare_xy(df: pd.DataFrame, feature_cols: list[str], categorical_cols: list[str]):
    X = df[feature_cols].copy()
    for col in categorical_cols:
        X[col] = X[col].astype('category')
    y = df['bike_change'].copy()
    return X, y

def build_lgbm_model():
    return LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        objective='regression',
    )


In [4]:
results = []

for model_name, feature_cols, categorical_cols in [
    ('LightGBM_RMSE_WithCluster', FEATURE_COLUMNS_WITH_CLUSTER, CATEGORICAL_WITH_CLUSTER),
    ('LightGBM_RMSE_NoCluster', FEATURE_COLUMNS_NO_CLUSTER, CATEGORICAL_NO_CLUSTER),
]:
    X_train, y_train = prepare_xy(train_2023, feature_cols, categorical_cols)
    X_valid, y_valid = prepare_xy(valid_2024, feature_cols, categorical_cols)
    X_full, y_full = prepare_xy(full_train, feature_cols, categorical_cols)
    X_test, y_test = prepare_xy(test_df, feature_cols, categorical_cols)

    model = build_lgbm_model()
    model.fit(X_train, y_train, categorical_feature=categorical_cols)
    results.append(evaluate_predictions(model_name, 'train_2023', y_train, model.predict(X_train)))
    results.append(evaluate_predictions(model_name, 'validation_2024', y_valid, model.predict(X_valid)))

    model = build_lgbm_model()
    model.fit(X_full, y_full, categorical_feature=categorical_cols)
    results.append(evaluate_predictions(model_name, 'train_full_refit', y_full, model.predict(X_full)))
    results.append(evaluate_predictions(model_name, 'test_2025_refit', y_test, model.predict(X_test)))

results_df = pd.DataFrame(results)
results_df.to_csv(METRICS_PATH, index=False, encoding='utf-8-sig')
print('saved:', METRICS_PATH)
results_df


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006024 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1956
[LightGBM] [Info] Number of data points in the train set: 128880, number of used features: 35
[LightGBM] [Info] Start training from score 0.000016


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.017769 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1994
[LightGBM] [Info] Number of data points in the train set: 260640, number of used features: 35
[LightGBM] [Info] Start training from score -0.000004


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006200 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1950
[LightGBM] [Info] Number of data points in the train set: 128880, number of used features: 34
[LightGBM] [Info] Start training from score 0.000016


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.018273 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1988
[LightGBM] [Info] Number of data points in the train set: 260640, number of used features: 34
[LightGBM] [Info] Start training from score -0.000004


saved: /Users/cheng80/Desktop/ddri_work/works/07_prediction_bike_change/output/data/ddri_bike_change_rep15_cluster_effect_metrics.csv


,model,split,rmse,mae,r2,sign_accuracy
0,LightGBM_RMSE_WithCluster,train_2023,0.9082,0.5852,0.6287,0.8988
1,LightGBM_RMSE_WithCluster,validation_2024,1.0055,0.6099,0.5414,0.8966
2,LightGBM_RMSE_WithCluster,train_full_refit,0.9272,0.5889,0.6115,0.8942
3,LightGBM_RMSE_WithCluster,test_2025_refit,0.8992,0.5482,0.5490,0.8908
4,LightGBM_RMSE_NoCluster,train_2023,0.9079,0.5853,0.6290,0.8977
5,LightGBM_RMSE_NoCluster,validation_2024,1.0048,0.6098,0.5420,0.8956
6,LightGBM_RMSE_NoCluster,train_full_refit,0.9276,0.5888,0.6112,0.8964
7,LightGBM_RMSE_NoCluster,test_2025_refit,0.9001,0.5485,0.5482,0.8895


In [5]:
results_df[results_df['split'] == 'test_2025_refit'].sort_values('rmse').reset_index(drop=True)


,model,split,rmse,mae,r2,sign_accuracy
0,LightGBM_RMSE_WithCluster,test_2025_refit,0.8992,0.5482,0.5490,0.8908
1,LightGBM_RMSE_NoCluster,test_2025_refit,0.9001,0.5485,0.5482,0.8895
